# EloSense: Chess Elo Prediction from Game Metadata

Can we guess a player's rating band just from how a game was played (time control, result type, format), without looking at their actual rating?

## 1. Setup & Data Loading

In [ ]:
import pandas as pd

DATA_PATH = "../data/club_games_data.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.dtypes)
df.head()

## 2. Data Quality Checks

In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

# Check whether each game is duplicated as a mirrored white/black row
print(df["pgn"].duplicated().sum())

### Leakage decision

Goal: predict a rating band (not the exact rating), using only game format and the opponent's rating band. Keeps this a classification task and avoids the trivial version of the problem.

Columns dropped from features, and why:

- `white_username`, `black_username`, `white_id`, `black_id`: these identify the player. A model could just memorize a specific person's rating instead of learning from the game itself.
- `pgn`, `fen`: move-level data, saving this for a later version.
- Opponent's exact rating: only using their rating band, so the task isn't trivial.
- The target player's own past rating isn't even in this dataset per row, so nothing to drop there.

Columns kept as features: `time_class`, `time_control`, `rules`, `rated`, `white_result`/`black_result`, and opponent rating band.

## 3. EDA: Rating Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["white_rating"], bins=50)
axes[0].set_title("White rating distribution")
axes[0].set_xlabel("Rating")

axes[1].hist(df["black_rating"], bins=50)
axes[1].set_title("Black rating distribution")
axes[1].set_xlabel("Rating")

plt.tight_layout()
plt.show()

## 4. EDA: Time Class and Result Breakdowns

In [ ]:
print(df["time_class"].value_counts())
print()
print(df["white_result"].value_counts())

bins = [0, 1000, 1400, 1800, 5000]
labels = ["<1000", "1000-1400", "1400-1800", "1800+"]
df["white_band"] = pd.cut(df["white_rating"], bins=bins, labels=labels)

top_results = ["checkmated", "resigned", "timeout", "win", "agreed"]
sub = df[df["white_result"].isin(top_results)]
pd.crosstab(sub["white_band"], sub["white_result"], normalize="index").round(3)

### Findings

- Blitz is by far the most common format (~29k games), followed by bullet (~22.5k), rapid (~13k), and daily is rare (~2k).
- Win rate for white climbs steadily with rating band: 49.3% under 1000, up to 57.5% at 1800+.
- Getting checkmated drops as rating rises: 15.3% under 1000 vs 10.7% at 1800+. Stronger players avoid getting mated more often.
- Losing on time also drops with rating: 17.9% under 1000 vs 13.8% at 1800+, suggesting better time management at higher levels.
- Resigning stays roughly flat across bands (~17-19%), so how often someone resigns isn't a strong skill signal on its own.

## 5. Rating Band Bucketing

In [ ]:
RATING_BINS = [0, 1000, 1400, 1800, 5000]
RATING_LABELS = ["<1000", "1000-1400", "1400-1800", "1800+"]

def rating_to_band(rating):
    return pd.cut([rating], bins=RATING_BINS, labels=RATING_LABELS)[0]

df["white_band"] = pd.cut(df["white_rating"], bins=RATING_BINS, labels=RATING_LABELS)
df["black_band"] = pd.cut(df["black_rating"], bins=RATING_BINS, labels=RATING_LABELS)

df[["white_rating", "white_band", "black_rating", "black_band"]].head()

## 6. Feature Encoding

In [ ]:
def result_to_outcome(result):
    if result == "win":
        return "win"
    if result in ["agreed", "repetition", "stalemate", "insufficient", "timevsinsufficient", "50move"]:
        return "draw"
    return "loss"

def result_to_reason(result):
    if result in ["checkmated", "resigned", "timeout", "abandoned"]:
        return result
    if result == "win":
        return "win"
    return "draw"

df["result_outcome"] = df["white_result"].apply(result_to_outcome)
df["result_reason"] = df["white_result"].apply(result_to_reason)

df[["white_result", "result_outcome", "result_reason"]].head()

In [ ]:
# Feature columns:
# - time_class, rules: game format
# - rated: whether the game affected Elo
# - result_outcome, result_reason: how the game ended (derived above)
# - black_band: opponent's rating band, not their exact rating (see leakage decision)
# Target: white_band

feature_cols = ["time_class", "rules", "rated", "result_outcome", "result_reason", "black_band"]

X = pd.get_dummies(df[feature_cols], columns=["time_class", "rules", "result_outcome", "result_reason", "black_band"])
y = df["white_band"]

print(X.shape, y.shape)
X.head()